# Hyperparameter Tuning — SimpleRNN on IMDB

This notebook follows the same IMDB + SimpleRNN approach as the original `simplernn` notebook, but uses **SciKeras + GridSearchCV** to search for better hyperparameters.

We will tune:
- `neurons` → number of SimpleRNN units
- `layers` → number of stacked SimpleRNN layers
- `embedding_dim` → size of each word embedding vector
- `batch_size` → number of reviews processed before one weight update
- `epochs` → maximum number of training passes


In [1]:
# If SciKeras is not installed, run this once in your environment:
# !pip install scikeras

import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.callbacks import EarlyStopping

from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV


In [2]:
# Load the IMDB dataset
max_features = 10000  # vocabulary size

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

print(f'Training data shape: {x_train.shape}')
print(f'Testing data shape: {x_test.shape}')


Training data shape: (25000,)
Testing data shape: (25000,)


In [3]:
# Make every review the same length
maxlen = 500

x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)

print(x_train.shape)


(25000, 500)


## Build a tunable SimpleRNN

Important correction from the earlier notebook: `layers` should control the **number of SimpleRNN layers**, not the number of Dense layers.

When we stack RNN layers, every RNN except the final one must use `return_sequences=True`, so it passes the complete sequence to the next RNN layer.

In [4]:
def create_model(neurons=128, layers=1, embedding_dim=128):
    model = Sequential()

    # Embedding layer
    model.add(Embedding(max_features, embedding_dim, input_length=maxlen))

    # SimpleRNN layers
    for i in range(layers):
        # All RNN layers except the last must return the sequence
        return_sequences = (i < layers - 1)
        model.add(SimpleRNN(neurons, activation='relu',
                            return_sequences=return_sequences))

    # Output layer for binary sentiment classification
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [5]:
# Create the SciKeras wrapper
model = KerasClassifier(
    model=create_model,
    verbose=0
)

model


,model,<function cre...001746A279B20>
,verbose,0
,build_fn,None
,warm_start,False
,random_state,None
,optimizer,'rmsprop'
,loss,None
,metrics,None
,batch_size,None
,validation_batch_size,None
,callbacks,None


## Define the hyperparameter grid

GridSearchCV will try every combination below and evaluate each combination using 3-fold cross-validation.

For example, `neurons=64` and `layers=2` creates a different RNN architecture from `neurons=128` and `layers=1`.

In [6]:
param_grid = {
    # 'model__neurons': [64, 128],
    'model__layers': [1, 2],
    # 'model__embedding_dim': [64, 128],
    # 'batch_size': [32, 64],
    'epochs': [5, 10]
}

param_grid


{'model__layers': [1, 2], 'epochs': [5, 10]}

## Run GridSearchCV

⚠️ This is computationally expensive because the IMDB dataset contains 25,000 training reviews and RNNs process sequences of length 500.

The grid above contains 32 combinations, and `cv=3` means each combination is trained 3 times.

In [7]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=1,
    verbose=2
)

grid_result = grid.fit(
    x_train,
    y_train,
    validation_split=0.2,
    callbacks=[early_stopping]
)


Fitting 3 folds for each of 4 candidates, totalling 12 fits




[CV] END ..........................epochs=5, model__layers=1; total time= 4.5min
[CV] END ..........................epochs=5, model__layers=1; total time= 3.5min
[CV] END ..........................epochs=5, model__layers=1; total time=17.6min
[CV] END ..........................epochs=5, model__layers=2; total time= 5.8min
[CV] END ..........................epochs=5, model__layers=2; total time=37.9min
[CV] END ........................epochs=5, model__layers=2; total time=2220.5min
[CV] END .........................epochs=10, model__layers=1; total time=23.0min
[CV] END .........................epochs=10, model__layers=1; total time=10.4min
[CV] END .........................epochs=10, model__layers=1; total time= 5.5min
[CV] END .........................epochs=10, model__layers=2; total time=35.3min
[CV] END .........................epochs=10, model__layers=2; total time=11.6min
[CV] END .........................epochs=10, 

In [8]:
print('Best parameters:')
print(grid_result.best_params_)

print('\nBest cross-validation accuracy:')
print(grid_result.best_score_)


Best parameters:
{'epochs': 5, 'model__layers': 2}

Best cross-validation accuracy:
0.7483256757676114


In [9]:
# View all combinations ranked by validation score
import pandas as pd

results = pd.DataFrame(grid_result.cv_results_)
results = results.sort_values('rank_test_score')

results[[
    'rank_test_score',
    'mean_test_score',
    'std_test_score',
    # 'param_model__neurons',
    'param_model__layers',
    # 'param_model__embedding_dim',
    # 'param_batch_size',
    'param_epochs'
]].head(10)


,rank_test_score,mean_test_score,std_test_score,param_model__layers,param_epochs
1,1,0.748326,0.101032,2,5
0,2,0.738237,0.057393,1,5
3,3,0.731797,0.116642,2,10
2,4,0.697516,0.140393,1,10


## Evaluate the best model on the test set

`best_estimator_` is the model created using the best hyperparameter combination found by GridSearchCV.

In [10]:
best_model = grid_result.best_estimator_

test_loss, test_accuracy = best_model.model_.evaluate(x_test, y_test, verbose=1)

print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')


782/782 [==============================] - 40s 51ms/step - loss: nan - accuracy: 0.5000
Test Loss: nan
Test Accuracy: 0.5000


## Final result

GridSearchCV searches the specified combinations, while cross-validation evaluates each combination.

**Grid Search → searches for the best settings 🔍**

**Cross-Validation → checks how reliably each setting performs 📊**